In [ ]:
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from collections import defaultdict
from torch.utils.data import DataLoader, Subset

import src.data_cleaning_and_preprocessing._model_ready_dataset as model_ready_dataset
from src.dataset.tiny_imagenet_dataset import TinyImageNetLoader
from archive.py_file.pc_6_classify import HierarchicalPCCNN


SEED = 42
BATCH_SIZE = 16

NUM_CLASSES = 20
IMAGES_PER_CLASS = 100

INFERENCE_STEPS = 100

PC_EPOCHS = 10
CLASSIFICATION_EPOCHS = 10

CLASSIFIER_LEARNING_RATE = 0.01

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
cleaned_tiny_imagenet = model_ready_dataset.ModelReadyDatasetLoader(
    TinyImageNetLoader(seed=SEED),
    image_size=64
).load()

train_dataset = cleaned_tiny_imagenet[0]
test_dataset = cleaned_tiny_imagenet[1]

In [ ]:
selected_classes = list(range(NUM_CLASSES))

class_indices = defaultdict(list)

for i in range(len(train_dataset)):
    _, label = train_dataset[i]
    label = int(label)

    if label in selected_classes:
        class_indices[label].append(i)


generator = torch.Generator()
generator.manual_seed(SEED)

indices = []

for class_id in selected_classes:
    available_indices = class_indices[class_id]

    if len(available_indices) < IMAGES_PER_CLASS:
        raise ValueError(
            f"Class {class_id} has only {len(available_indices)} images, "
            f"but {IMAGES_PER_CLASS} were requested."
        )

    perm = torch.randperm(len(available_indices), generator=generator)[:IMAGES_PER_CLASS]
    selected = [available_indices[i] for i in perm.tolist()]
    indices.extend(selected)


train_subset = Subset(train_dataset, indices)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator
)

print("Total Tiny ImageNet training images:", len(train_dataset))
print("Selected classes:", selected_classes)
print("Number of selected classes:", len(selected_classes))
print("Images per class:", IMAGES_PER_CLASS)
print("Training images:", len(train_subset))
print("Training batches:", len(train_loader))

In [ ]:
test_class_indices = defaultdict(list)

for i in range(len(test_dataset)):
    _, label = test_dataset[i]
    label = int(label)

    if label in selected_classes:
        test_class_indices[label].append(i)


test_generator = torch.Generator()
test_generator.manual_seed(SEED)

test_indices = []

IMAGES_PER_CLASS = 10

for class_id in selected_classes:
    available_indices = test_class_indices[class_id]

    if len(available_indices) < IMAGES_PER_CLASS:
        raise ValueError(
            f"Test class {class_id} has only {len(available_indices)} images, "
            f"but {IMAGES_PER_CLASS} were requested."
        )

    perm = torch.randperm(len(available_indices), generator=test_generator)[:IMAGES_PER_CLASS]
    selected = [available_indices[i] for i in perm.tolist()]
    test_indices.extend(selected)


test_subset = Subset(test_dataset, test_indices)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Total Tiny ImageNet test images:", len(test_dataset))
print("Selected test classes:", selected_classes)
print("Number of selected classes:", len(selected_classes))
print("Images per class:", IMAGES_PER_CLASS)
print("Test images:", len(test_subset))
print("Test batches:", len(test_loader))

In [ ]:
model = HierarchicalPCCNN(
    num_classes=NUM_CLASSES,
    inference_steps=INFERENCE_STEPS
).to(device)

print(model)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
autograd_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total unique parameters:", total_params)
print("Autograd-enabled parameters:", autograd_params)

In [ ]:
@torch.no_grad()
def train_model(model, train_loader, device, classifier_learning_rate):

    total_image_error = 0.0
    total_r1_error = 0.0

    total_classification_loss = 0.0
    total_classification_correct = 0

    total_images = 0

    for images, labels in train_loader:

        for image, label in zip(images, labels):

            result = model.train_step(
                image,
                label,
                classifier_learning_rate=classifier_learning_rate
            )

            # Predictive-coding metrics
            total_image_error += result["image_error"]
            total_r1_error += result["r1_error"]

            # Classification metrics
            total_classification_loss += result["classification_loss"]
            total_classification_correct += result["classification_correct"]

            total_images += 1

    mean_image_error = total_image_error / total_images
    mean_r1_error = total_r1_error / total_images

    mean_classification_loss = total_classification_loss / total_images
    classification_accuracy = 100.0 * total_classification_correct / total_images

    return {
        "image_error": mean_image_error,
        "r1_error": mean_r1_error,
        "classification_loss": mean_classification_loss,
        "classification_accuracy": classification_accuracy
    }

In [ ]:
@torch.no_grad()
def test_model(model, test_loader, device):

    total_image_error = 0.0
    total_r1_error = 0.0

    total_classification_loss = 0.0
    total_classification_correct = 0

    total_images = 0

    for images, labels in test_loader:

        for image, label in zip(images, labels):

            image = image.to(device)
            label = label.to(device, dtype=torch.long)

            # ================================================
            # PREDICTIVE-CODING INFERENCE
            # ================================================

            r1, r2, steps, converged = model.infer(image)

            # ================================================
            # PREDICTIVE-CODING TEST METRICS
            # ================================================

            e0, e1, _ = model.prediction_errors(image, r1, r2)

            image_error = torch.sqrt(torch.mean(e0 ** 2)).item()
            r1_error = torch.sqrt(torch.mean(e1 ** 2)).item()

            # ================================================
            # CLASSIFICATION USING SAME r2
            # ================================================

            logits = model.classify_from_r2(r2)
            probabilities = torch.softmax(logits, dim=1)

            true_probability = probabilities[0, label.item()]
            classification_loss = -torch.log(true_probability + 1e-8).item()

            prediction = logits.argmax(dim=1).item()
            correct = int(prediction == label.item())

            # ================================================
            # ACCUMULATE
            # ================================================

            total_image_error += image_error
            total_r1_error += r1_error

            total_classification_loss += classification_loss
            total_classification_correct += correct

            total_images += 1

    mean_image_error = total_image_error / total_images
    mean_r1_error = total_r1_error / total_images

    mean_classification_loss = total_classification_loss / total_images
    classification_accuracy = 100.0 * total_classification_correct / total_images

    return {
        "image_error": mean_image_error,
        "r1_error": mean_r1_error,
        "classification_loss": mean_classification_loss,
        "classification_accuracy": classification_accuracy
    }

In [ ]:
TRAINING_EPOCHS = 10
CLASSIFIER_LEARNING_RATE = 0.01

training_history = []

for epoch in tqdm(range(TRAINING_EPOCHS), desc="PC-CNN Training"):
    # ====================================================
    # TRAIN
    # ====================================================

    train_results = train_model(
        model,
        train_loader,
        device,
        CLASSIFIER_LEARNING_RATE
    )

    # ====================================================
    # TEST
    # ====================================================

    test_results = test_model(
        model,
        test_loader,
        device
    )

    # ====================================================
    # SAVE HISTORY
    # ====================================================

    training_history.append({
        "epoch": epoch + 1,

        "train_image_error": train_results["image_error"],
        "train_r1_error": train_results["r1_error"],
        "train_classification_loss": train_results["classification_loss"],
        "train_classification_accuracy": train_results["classification_accuracy"],

        "test_image_error": test_results["image_error"],
        "test_r1_error": test_results["r1_error"],
        "test_classification_loss": test_results["classification_loss"],
        "test_classification_accuracy": test_results["classification_accuracy"]
    })

    # ====================================================
    # PRINT
    # ====================================================

    print(
        f"\nEpoch {epoch + 1}/{TRAINING_EPOCHS} | "
        f"Train Image Error: {train_results['image_error']:.6f} | "
        f"Train r1 Error: {train_results['r1_error']:.6f} | "
        f"Train Class Loss: {train_results['classification_loss']:.4f} | "
        f"Train Accuracy: {train_results['classification_accuracy']:.2f}%"
    )

    print(
        f"Test Image Error: {test_results['image_error']:.6f} | "
        f"Test r1 Error: {test_results['r1_error']:.6f} | "
        f"Test Class Loss: {test_results['classification_loss']:.4f} | "
        f"Test Accuracy: {test_results['classification_accuracy']:.2f}%"
    )

In [ ]:
epochs = [item["epoch"] for item in training_history]

train_image_error = [item["train_image_error"] for item in training_history]
test_image_error = [item["test_image_error"] for item in training_history]

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_image_error, marker="o", label="Train Image Error")
plt.plot(epochs, test_image_error, marker="o", label="Test Image Error")

plt.xlabel("Epoch")
plt.ylabel("RMS Prediction Error")
plt.title("PC-CNN Image Prediction Error")
plt.xticks(epochs)
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
train_r1_error = [item["train_r1_error"] for item in training_history]
test_r1_error = [item["test_r1_error"] for item in training_history]

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_r1_error, marker="o", label="Train r1 Error")
plt.plot(epochs, test_r1_error, marker="o", label="Test r1 Error")

plt.xlabel("Epoch")
plt.ylabel("RMS Prediction Error")
plt.title("PC-CNN r1 Prediction Error")
plt.xticks(epochs)
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
train_classification_loss = [item["train_classification_loss"] for item in training_history]
test_classification_loss = [item["test_classification_loss"] for item in training_history]

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_classification_loss, marker="o", label="Train Classification Loss")
plt.plot(epochs, test_classification_loss, marker="o", label="Test Classification Loss")

plt.xlabel("Epoch")
plt.ylabel("Classification Loss")
plt.title("PC-CNN Classification Loss")
plt.xticks(epochs)
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
train_classification_accuracy = [item["train_classification_accuracy"] for item in training_history]
test_classification_accuracy = [item["test_classification_accuracy"] for item in training_history]

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_classification_accuracy, marker="o", label="Train Accuracy")
plt.plot(epochs, test_classification_accuracy, marker="o", label="Test Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("PC-CNN Classification Accuracy")
plt.xticks(epochs)
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.3)

plt.show()